**Import Key Libraries**

In [ ]:
from pathlib import Path
import sys
import tempfile
import pandas as pd

SCRAPING_DIR = Path.cwd()
if (SCRAPING_DIR / "web-scrapping").is_dir():
    SCRAPING_DIR = SCRAPING_DIR / "web-scrapping"
if not (SCRAPING_DIR / "flipkart_data_extraction.ipynb").exists():
    raise RuntimeError("Open the notebook from the project root or web-scrapping folder.")
RUNS_DIR = SCRAPING_DIR / "refresh-runs"
RUNS_DIR.mkdir(exist_ok=True)
RUN_DIR = Path(tempfile.mkdtemp(prefix="scrape-", dir=RUNS_DIR))
sys.path.insert(0, str(SCRAPING_DIR))
print("Staging scrape in:", RUN_DIR)


### Step 1: Load existing catalog product links


In [ ]:
# Refresh the same products the chatbot already knows, preserving catalog coverage.
# New keyword searches could return an unrelated catalog and are not a price refresh.
import sqlite3
from flipkart_scraper import stage_products

db_path = SCRAPING_DIR.parent / "app" / "db.sqlite"
with sqlite3.connect(f"{db_path.resolve().as_uri()}?mode=ro", uri=True) as connection:
    all_product_links = [row[0] for row in connection.execute("SELECT product_link FROM product ORDER BY rowid")]
if not all_product_links:
    raise ValueError("Existing catalog is empty; no products to refresh.")
pd.DataFrame(all_product_links, columns=["product_links"]).to_csv(RUN_DIR / "flipkart_product_links.csv", index=False)
print(f"Refreshing {len(all_product_links)} existing product pages.")


### Step 2: Extract current structured product data, validate, and publish


In [ ]:
# Fetch every product; failed or invalid products keep their previous records.
csv_path, manifest_path = stage_products(all_product_links, RUN_DIR)
from csv_to_sqlite import merge_refresh
summary = merge_refresh(csv_path, db_path, manifest_path=manifest_path)
print(f"Updated {summary['updated_products']} products; retained {summary['retained_products']} previous records. Database backed up.")
